# Strict Pure-Exogenous HDM Error Pipeline

Notebook runner separato per area tematica. Carica le definizioni dal notebook-libreria e poi esegue smoke run, batch run e diagnostica.

## Base Imports

Import minimi per bootstrap, configurazione e visualizzazione.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Config Block

Tutta la configurazione utente è concentrata qui.

In [ ]:
DATA_DIR = Path('/Users/nicolocaron/Desktop/MASTER PROJECT/data/per_station')
DTU10_DIR = Path('/Volumes/DTU10_TIDEMODEL/OCEAN_TIDE_GRIDS/FES2004_FORMAT')
USE_DTU10_PREDICTOR = False

STATIONS_TO_RUN = ('Sonderborg', 'Assens', 'Bagenkop', 'Kobenhavn', 'Koge', 'Gedser')
HORIZONS = (1, 3, 6, 12, 24)
LAGS = (0, 1, 2, 3, 6, 12, 24, 48, 72, 96)
RIDGE_ALPHAS = (0.1, 1.0, 10.0, 100.0)
TRAIN_FRAC = 0.8
INTERPOLATE_EXOG_LIMIT = 6

SMOKE_STATION = 'Kobenhavn'
SMOKE_HORIZON = 12
HEATMAP_MODEL = 'ridge_exog'
DETAIL_STATION = 'Kobenhavn'
DETAIL_HORIZON = 12
DETAIL_MODEL = 'ridge_exog'


## Scientific Setup

Questo è un **causal input-output benchmark**. Il benchmark principale è **puramente esogeno**: i lag del target sono vietati nel benchmark principale. Ridge è usato per la collinearità dei forcing laggati; i coefficienti vanno interpretati come **empirical impulse responses**. L’obiettivo è quantificare la spiegabilità lineare prima dei modelli non lineari.

## Load Definitions Notebook

Questa cella esegue tutte le celle codice del notebook-libreria nella sessione corrente.

In [ ]:
def exec_notebook_code(path, scope):
    notebook = json.loads(Path(path).read_text())
    for cell in notebook['cells']:
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        if src.strip():
            exec(src, scope)
    return scope


exec_notebook_code('01_hdm_error_correction_thematic.ipynb', globals())


## Runtime Checks

Verifica rapido di path, dipendenze e costruzione della configurazione.

In [ ]:
def package_version(name):
    try:
        return getattr(__import__(name), '__version__', 'installed')
    except Exception as exc:
        return f'missing: {type(exc).__name__}'


cfg = PipelineConfig(
    data_dir=DATA_DIR,
    dtu10_dir=DTU10_DIR,
    use_dtu10_predictor=USE_DTU10_PREDICTOR,
    stations=STATIONS_TO_RUN,
    horizons=HORIZONS,
    lags=LAGS,
    train_frac=TRAIN_FRAC,
    ridge_alphas=RIDGE_ALPHAS,
    interpolate_exog_limit=INTERPOLATE_EXOG_LIMIT,
)
event_cfg = EventConfig()

package_checks = pd.DataFrame(
    [
        {'package': 'numpy', 'version': np.__version__},
        {'package': 'pandas', 'version': pd.__version__},
        {'package': 'matplotlib', 'version': plt.matplotlib.__version__},
        {'package': 'seaborn', 'version': package_version('seaborn')},
        {'package': 'statsmodels', 'version': package_version('statsmodels')},
        {'package': 'sklearn', 'version': package_version('sklearn')},
        {'package': 'sktime', 'version': package_version('sktime')},
        {'package': 'pyTMD', 'version': package_version('pyTMD') if USE_DTU10_PREDICTOR else 'optional-not-requested'},
    ]
)

print(f'Data dir exists: {cfg.data_dir.exists()} -> {cfg.data_dir}')
print(f'DTU10 dir exists: {cfg.dtu10_dir.exists()} -> {cfg.dtu10_dir}')
print(f'Stations in loaded notebook module: {list(STATIONS.keys())}')
package_checks


## Smoke Run

Esecuzione rapida su una sola stazione e un solo orizzonte.

In [ ]:
smoke_suite = run_station_horizon_suite(SMOKE_STATION, SMOKE_HORIZON, cfg, event_cfg)
smoke_suite['metrics']


## Batch Run

Run completo su tutte le stazioni e tutti gli orizzonti configurati.

In [ ]:
all_runs = run_all_stations(cfg, event_cfg)
print(f"Failures: {len(all_runs['failures'])}")
all_runs['metrics']


## Summary Tables

Tabella compatta delle metriche aggregate per stazione, orizzonte e modello.

In [ ]:
summary_metrics = all_runs['metrics'].sort_values(['station', 'horizon_h', 'model_name']).reset_index(drop=True)
summary_metrics


## Heatmaps

Visualizzazione sintetica di R2 e RMSE del modello scelto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
plot_metric_heatmap(all_runs['metrics'], metric='r2', model_name=HEATMAP_MODEL, ax=axes[0])
plot_metric_heatmap(all_runs['metrics'], metric='rmse_m', model_name=HEATMAP_MODEL, ax=axes[1])
plt.show()


## Model Comparison

Confronto tra benchmark strict pure-exog e auxiliary event-only.

In [ ]:
comparison_detail = all_runs['metrics'][
    ['station', 'horizon_h', 'model_name', 'strict_pure_exog', 'use_dtu10_predictor', 'r2', 'rmse_m', 'mae_m', 'bias_m', 'event_threshold_m']
].sort_values(['station', 'horizon_h', 'strict_pure_exog', 'model_name']).reset_index(drop=True)

comparison_summary = (
    all_runs['metrics']
    .groupby(['model_name', 'strict_pure_exog', 'use_dtu10_predictor'], as_index=False)[['r2', 'rmse_m', 'mae_m', 'bias_m']]
    .mean()
    .sort_values(['strict_pure_exog', 'model_name'])
)

print(comparison_summary.to_string(index=False))
comparison_detail


## Event Detail

Plot dettagliato di un evento per una stazione, orizzonte e modello selezionati.

In [ ]:
detail_suite = all_runs['suites'][(DETAIL_STATION, DETAIL_HORIZON)]
detail_result = detail_suite['results'][DETAIL_MODEL]
detail_event_df = detail_result.diagnostics.get('event_eval_df', detail_result.eval_df)
detail_event_id = int(detail_event_df['event_id'].dropna().iloc[0]) if 'event_id' in detail_event_df.columns and not detail_event_df.empty else None

plot_event_window(detail_result, event_id=detail_event_id)
plt.show()


## Residual Diagnostics

Distribuzione, ACF, QQ plot e serie temporale dei residui corretti.

In [ ]:
plot_residual_diagnostics(detail_result)
plt.show()


## Coefficients And Impulse Responses

Tabella coefficienti e grafico dei lag dominanti del modello scelto.

In [ ]:
coef_focus = (
    all_runs['coefficients']
    .loc[
        (all_runs['coefficients']['station'] == DETAIL_STATION)
        & (all_runs['coefficients']['horizon_h'] == DETAIL_HORIZON)
        & (all_runs['coefficients']['model_name'] == DETAIL_MODEL)
    ]
    .sort_values(['variable', 'lag_h'])
    .reset_index(drop=True)
)

print(coef_focus.head(20).to_string(index=False))
plot_impulse_responses(coef_focus, station=DETAIL_STATION, horizon_h=DETAIL_HORIZON, model_name=DETAIL_MODEL)
plt.show()


## Short Interpretation

Il runner rimane fedele al benchmark causale input-output: solo forcing esogeni laggati nel benchmark principale, threshold eventi stimato solo sul train, tuning Ridge train-only e variante `ridge_event_only_aux` isolata dal benchmark strict.